# Lab 10: predict the verdict of a reply

**50–60 minutes.** Work with your team. Make a copy named `lab10_mine.ipynb`. Switch who types halfway through.

On Monday you predicted the verdict of AITA posts. Here you predict the verdict of replies to AITA posts, after removing the NTA or YTA they start with. You compare three methods on the same replies: always guessing NTA, a classifier, and Gemini.

You need your Gemini key in the `.env` file.

In [ ]:
from pathlib import Path
import json
import logging
import os
import time
import warnings

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

for folder in (Path.cwd(), *Path.cwd().parents):
    env_path = folder / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break

key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"

logging.getLogger("google_genai.types").setLevel(logging.ERROR)  # a warning google-genai prints on every call
warnings.filterwarnings("ignore", message="Unknown solver options")  # a harmless warning from LogisticRegression
pd.set_option("display.max_colwidth", 300)  # show more of each text
print("Key loaded.")

## 1. Load the replies and set a baseline

In [ ]:
url = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/docs/interactives/data/week08/aita_replies.csv"
replies = pd.read_csv(url).rename(columns={"idstr": "id"})
replies["verdict"] = replies["body"].str.upper().str.extract(r"^(NTA|YTA)\b")[0]
replies = replies.dropna(subset=["verdict"]).reset_index(drop=True)
replies["verdict"].value_counts()  # Check: NTA 657, YTA 378

Every reply states its verdict, so a model could simply read it. Remove the verdict words NTA, YTA, ESH, NAH, and INFO wherever they appear in a reply, and save the result in `replies["text"]`.

*Hint: in a regular expression, `|` means "or".*

In [ ]:
replies["text"] = replies["body"].str.replace(r"\b(____)\b", "", regex=True, case=False)
replies["text"].str.contains(r"\b(NTA|YTA|ESH|NAH|INFO)\b", case=False).sum()  # replies that still contain a verdict word. Check: 0

In [ ]:
replies[["body", "text"]].head(3)

In [ ]:
# The accuracy of always guessing NTA. Check: 0.63

## 2. Train a classifier

Split the replies as on Monday.

In [ ]:
train, test = train_test_split(replies, test_size=0.25, stratify=replies["verdict"], random_state=211)
len(train), len(test)  # Check: (776, 259)

Train a TF-IDF and logistic regression classifier on `train["text"]`, with `stop_words="english"`, `min_df=5`, and `class_weight="balanced"`, as on Monday. Save its predictions for the test replies in `test["prediction"]`.

In [ ]:
# Fit TF-IDF and the model on train, predict test, and show the accuracy. Check: 0.58

In [ ]:
# Make a confusion table. Check: 39 YTA replies predicted YTA

The classifier does worse than always guessing NTA. Read three test replies it got wrong. Why might word weights fail to find the verdict of a reply?

Your answer:

## 3. Ask Gemini

Gemini labels 20 of the test replies, 10 of each verdict. The system instruction tells it that the verdict was removed. Read the cell, then run it.

In [ ]:
sample = test.groupby("verdict").sample(n=10, random_state=211)

SYSTEM = """You read replies to posts on the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong. The verdict at the start of each reply has been removed.

Predict the verdict the reply gave:
- NTA: the reply says the poster was not in the wrong.
- YTA: the reply says the poster was in the wrong.

Also give a reason of one short sentence."""

CONFIG = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    response_mime_type="application/json",
    response_schema={
        "type": "object",
        "properties": {
            "llm_label": {"type": "string", "enum": ["NTA", "YTA"]},
            "llm_reason": {"type": "string"},
        },
        "required": ["llm_label", "llm_reason"],
    },
)
len(sample)  # Check: 20

In [ ]:
# Send one prompt, and try again after error 429 or 503.
def ask(prompt, config, max_tries=4):
    wait = 10
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
            time.sleep(6)  # stay under the free limit of calls per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in {429, 503} or attempt == max_tries:
                raise
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2

In [ ]:
# Label the 20 replies. Each answer is saved right away; if the cell stops, run it again. Takes about 3 minutes.
GEMINI_PATH = Path("lab10_gemini_labels.csv")
done = set(pd.read_csv(GEMINI_PATH)["id"]) if GEMINI_PATH.exists() else set()
for row in sample.itertuples():
    if row.id in done:
        continue
    answer = ask(f"Reply: {row.text}", CONFIG)
    line = pd.DataFrame([{"id": row.id, **answer, "model": GEMINI_MODEL}])
    line.to_csv(GEMINI_PATH, mode="a", header=not GEMINI_PATH.exists(), index=False)
    done.add(row.id)
    print(len(done), "of", len(sample), "done")

## 4. Compare the three methods

Read the file into `gemini` and merge it with `sample` on `id`, so that each row has the verdict, the classifier's prediction, and Gemini's label.

In [ ]:
# Make gemini. Check: 20 rows

In [ ]:
# Print the accuracy of always NTA, the classifier, and Gemini on the 20 replies. Check: NTA 0.5, classifier 0.35, Gemini about 0.85 (it can change a little between runs)

In [ ]:
# Make a confusion table for Gemini.

In [ ]:
# Show the text, verdict, and Gemini's label and reason for the replies Gemini got wrong.

Read the replies Gemini got wrong. Did Gemini misread them, or is the verdict hard to see without the first word? Would you trust Gemini's labels for the other 1,000 replies?

Your answer:

## 5. Describe what you found

Write three or four sentences: compare the three methods on the replies, and compare this with Monday's results on the posts. Say what the comparison rests on (how many replies, which baseline) and give one reason to be careful with the result.

Your finding:

Check that your key does not appear in any cell or output. Then save the notebook.

## If you have time

Change one thing in `SYSTEM`, such as describing how an NTA reply and a YTA reply usually sound, or adding an `unclear` label (add it to the schema too). Label the same 20 replies again with a new file, `lab10_gemini_labels_v2.csv`, and count how many labels changed.

In [ ]:
# Your turn